# Browse and scrape

**The job.** Take a web page. Pull out the products. Save them as clean rows.

We build a page first so this notebook runs the same way every time. At the end
there is a one-line change that points the same graph at a real URL. The graph
does not change. Only the function behind one step does.

**In:** an HTML page.
**Out:** a list of product rows.
**Files:** the raw page, the rows as JSONL, and a run receipt.

In [1]:
try:
    import browsergraph  # noqa: F401
except ImportError:
    %pip install -q "browsergraph @ git+https://github.com/aidonerightcorp/browsergraph.git"

import json, pathlib
from dataclasses import replace

from browsergraph import execute, viz
from browsergraph.compile import compile_route
from browsergraph.manifest import NodeManifest, PortSpec
from browsergraph.workbench import Edge, NodeCandidate, StageDefinition, WorkbenchDefinition

# A fresh folder each run. Left-over files from a previous run make the "what
# did this produce" list a lie, and that list is half the point here.
import shutil
WORK = pathlib.Path("work")
shutil.rmtree(WORK, ignore_errors=True)
WORK.mkdir()

# These come from the library rather than being redefined in every notebook.
# They used to be thirty lines pasted into each one, which meant anyone copying
# a notebook to start a project got helpers that did not exist in browsergraph.
from browsergraph.quick import chain, fanin, fanout, link, node, problems, step
from browsergraph.quick import graph as _graph
from browsergraph.quick import subgraph  # noqa: F401  (used by later notebooks)

# The notebooks kept the older names, and `build` also prints what is wrong
# rather than raising — in a notebook the complaint is the lesson.
stage = step

def build(title, task, stages, nodes, edges=()):
    bench = _graph(title, task, stages, nodes, edges)
    print("problems:", problems(bench) or "none")
    return bench

print("ready")

ready


## The input

A page with four products. One has a missing price and one has a price written
in a different style. Real pages are like this. A scraper that only works on
tidy pages is not finished.

In [2]:
PAGE = """<!doctype html><html><body>
<h1>Camping gear</h1>
<ul class="products">
  <li class="product"><span class="name">Tent 2P</span><span class="price">$189.00</span><span class="sku">TN-2P</span></li>
  <li class="product"><span class="name">Sleeping bag</span><span class="price">USD 74.50</span><span class="sku">SB-01</span></li>
  <li class="product"><span class="name">Camp stove</span><span class="price"></span><span class="sku">CS-77</span></li>
  <li class="product"><span class="name">Head torch</span><span class="price">$21</span><span class="sku">HT-03</span></li>
</ul></body></html>"""

source = WORK / "page.html"
source.write_text(PAGE)
print(f"wrote {source} ({source.stat().st_size} bytes)")
print(PAGE[:180], "...")

wrote work/page.html (593 bytes)
<!doctype html><html><body>
<h1>Camping gear</h1>
<ul class="products">
  <li class="product"><span class="name">Tent 2P</span><span class="price">$189.00</span><span class="sku">T ...


## The steps

Six steps. Each says what it needs and what it gives back. Nothing here says
*how* yet.

In [3]:
nodes = [
    node("read.file",     "payload.read",   [],                      [("out", "Bytes")]),
    node("read.http",     "payload.read",   [],                      [("out", "Bytes")],
         effects=("network.read",), permissions=("net.read",), runtime={"deterministic": False}),
    node("parse.html",    "parse.structure",[("in", "Bytes")],       [("out", "Blocks")]),
    node("locate.fields", "locate.values",  [("in", "Blocks")],      [("out", "Fields")]),
    node("clean.values",  "normalise",      [("in", "Fields")],      [("out", "Records")]),
    node("check.rows",    "verify",         [("in", "Records")],     [("kept", "Records"), ("dropped", "Records")]),
    node("save.jsonl",    "write",          [("in", "Records")],     [("out", "Receipt")],
         effects=("file.write",)),
]

stages = [
    stage("read",   "Read the page",   [],                  [("out", "Bytes")],   "payload.read",    ["read.file", "read.http"]),
    stage("parse",  "Parse the HTML",  [("in", "Bytes")],   [("out", "Blocks")],  "parse.structure", ["parse.html"]),
    stage("locate", "Find the fields", [("in", "Blocks")],  [("out", "Fields")],  "locate.values",   ["locate.fields"]),
    stage("clean",  "Clean the values",[("in", "Fields")],  [("out", "Records")], "normalise",       ["clean.values"]),
    stage("check",  "Check each row",  [("in", "Records")], [("kept", "Records"), ("dropped", "Records")], "verify", ["check.rows"]),
    stage("save",   "Save the rows",   [("in", "Records")], [("out", "Receipt")], "write",           ["save.jsonl"]),
]

edges = [Edge("read", "parse"), Edge("parse", "locate"), Edge("locate", "clean"),
         Edge("clean", "check"), Edge("check", "save", from_port="kept")]

bench = build("Scrape a product page",
              "Turn a page of products into clean rows.", stages, nodes, edges)
print("layers:", bench.layers())

problems: none
layers: [['read'], ['parse'], ['locate'], ['clean'], ['check'], ['save']]


Note the `check` step has **two** outputs: rows it kept and rows it dropped.
Only the kept ones go on to be saved. The dropped ones are still there to look
at. A scraper that quietly bins bad rows is how you find out months later.

In [4]:
viz.dag(bench)

Figure(svg='<svg viewBox="0 0 1170 226" width="1170" height="226" style="max-width:none" role="img"><defs><marker id="bg75614832-arrow" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-end"><path d="M0,0 L10,5 L0,10 z" fill="#8a93a0"/></marker></defs><text x="153.0" y="44" text-anchor="middle" font-size="10" fill="#68737f">layer 0</text><g><rect x="60" y="74.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="69" y="93.0" font-size="11.5" font-weight="700" fill="#22303f">Read the page</text><text x="69" y="108.0" font-size="9.5" fill="#68737f">2 candidates</text></g><text x="363.0" y="44" text-anchor="middle" font-size="10" fill="#68737f">layer 1</text><g><rect x="270" y="74.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="279" y="93.0" font-size="11.5" font-weight="700" fill="#22303f">Parse the HTML</text><text x="279" y="108.0" font-size="9.5" fill="#68737f">1 candidate</text></g><text x="573.0" y="44" text-anchor="middle" font-size="10" fill="#68737f">layer 2</text><g><rect x="480" y="74.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="489" y="93.0" font-size="11.5" font-weight="700" fill="#22303f">Find the fields</text><text x="489" y="108.0" font-size="9.5" fill="#68737f">1 candidate</text></g><text x="783.0" y="44" text-anchor="middle" font-size="10" fill="#68737f">layer 3</text><g><rect x="690" y="74.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="699" y="93.0" font-size="11.5" font-weight="700" fill="#22303f">Clean the values</text><text x="699" y="108.0" font-size="9.5" fill="#68737f">1 candidate</text></g><text x="993.0" y="44" text-anchor="middle" font-size="10" fill="#68737f">layer 4</text><g><rect x="900" y="74.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="909" y="93.0" font-size="11.5" font-weight="700" fill="#22303f">Check each row</text><text x="909" y="108.0" font-size="9.5" fill="#68737f">1 candidate</text></g><text x="1203.0" y="44" text-anchor="middle" font-size="10" fill="#68737f">layer 5</text><g><rect x="1110" y="74.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="1119" y="93.0" font-size="11.5" font-weight="700" fill="#22303f">Save the rows</text><text x="1119" y="108.0" font-size="9.5" fill="#68737f">1 candidate</text></g><path d="M246,100.0 C258.0,100.0 258.0,100.0 270,100.0" fill="none" stroke="#8a93a0" stroke-width="1.4" opacity=".75" marker-end="url(#bg75614832-arrow)"/><path d="M456,100.0 C468.0,100.0 468.0,100.0 480,100.0" fill="none" stroke="#8a93a0" stroke-width="1.4" opacity=".75" marker-end="url(#bg75614832-arrow)"/><path d="M666,100.0 C678.0,100.0 678.0,100.0 690,100.0" fill="none" stroke="#8a93a0" stroke-width="1.4" opacity=".75" marker-end="url(#bg75614832-arrow)"/><path d="M876,100.0 C888.0,100.0 888.0,100.0 900,100.0" fill="none" stroke="#8a93a0" stroke-width="1.4" opacity=".75" marker-end="url(#bg75614832-arrow)"/><path d="M1086,100.0 C1098.0,100.0 1098.0,100.0 1110,100.0" fill="none" stroke="#8a93a0" stroke-width="1.4" opacity=".75" marker-end="url(#bg75614832-arrow)"/><text x="1098.0" y="95.0" text-anchor="middle" font-size="9" fill="#68737f">kept</text></svg>', title='Scrape a product page — shape', note='a chain. Boxes in the same layer are independent and may run together; every arrow is a typed port-to-port connection.', width=1170, height=226)

## Now the actual code

One function per step. Plain Python. `html.parser` is in the standard library,
so there is nothing to install.

In [5]:
from html.parser import HTMLParser

class Collector(HTMLParser):
    """Grab the text inside every <span class="..."> we care about."""
    def __init__(self):
        super().__init__()
        self.rows, self.current, self.field = [], {}, None
    def handle_starttag(self, tag, attrs):
        classes = dict(attrs).get("class", "")
        if tag == "li" and "product" in classes:
            self.current = {}
        if tag == "span" and classes in ("name", "price", "sku"):
            self.field = classes
            self.current.setdefault(classes, "")
    def handle_endtag(self, tag):
        if tag == "li" and self.current:
            self.rows.append(self.current); self.current = {}
        if tag == "span":
            self.field = None
    def handle_data(self, text):
        if self.field:
            self.current[self.field] += text.strip()

def read_file():
    return source.read_bytes()

def read_http():
    """Fetch a real page over the network."""
    import urllib.request
    with urllib.request.urlopen(LIVE_URL, timeout=20) as response:
        return response.read()

def parse_html(**kw):
    collector = Collector()
    collector.feed(kw["in"].decode())
    return collector.rows

def locate_fields(**kw):
    return [{"name": r.get("name", ""), "price_text": r.get("price", ""),
             "sku": r.get("sku", "")} for r in kw["in"]]

def clean_values(**kw):
    """Prices come in three styles here. Turn them all into a number."""
    import re
    out = []
    for row in kw["in"]:
        digits = re.sub(r"[^0-9.]", "", row["price_text"])
        out.append({"sku": row["sku"], "name": row["name"],
                    "price": float(digits) if digits else None,
                    "currency": "USD" if row["price_text"] else ""})
    return out

def check_rows(**kw):
    """A row without a price is not a product row we can use."""
    kept = [r for r in kw["in"] if r["price"] is not None]
    dropped = [dict(r, why="no price on the page") for r in kw["in"] if r["price"] is None]
    return {"kept": kept, "dropped": dropped}

def save_jsonl(workspace, **kw):
    path = workspace / "products.jsonl"
    path.write_text("\n".join(json.dumps(r) for r in kw["in"]))
    return {"rows": len(kw["in"]), "path": str(path)}

runtime = execute.Runtime({
    "read.file": read_file, "read.http": read_http, "parse.html": parse_html,
    "locate.fields": locate_fields, "clean.values": clean_values,
    "check.rows": check_rows, "save.jsonl": save_jsonl,
})
print("nothing missing:", runtime.missing(compile_route(bench, {s.id: s.candidates[0] for s in bench.leaf_stages})) == [])

nothing missing: True


## Run it

In [6]:
route = {s.id: s.candidates[0] for s in bench.leaf_stages}
plan = compile_route(bench, route)
run = execute.run(plan, runtime, workspace=WORK)
print(run.text())

plan plan:c4bfdc35a412c87c7954c…
6 steps in 0.002s — ok
  ok   read             0.000s  read.file
  ok   parse            0.000s  parse.html
  ok   locate           0.000s  locate.fields
  ok   clean            0.000s  clean.values
  ok   check            0.000s  check.rows
  ok   save             0.000s  save.jsonl  [file.write]
  file /home/username/code_projects/repos/browsergraph/notebooks/work/products.jsonl  218 bytes  sha256:15ae8e21f…


## What came out

In [7]:
kept = run.values[("check", "kept")]
dropped = run.values[("check", "dropped")]

print(f"{'sku':<8}{'name':<16}{'price':>9}  currency")
for row in kept:
    print(f"{row['sku']:<8}{row['name']:<16}{row['price']:>9.2f}  {row['currency']}")

print(f"\ndropped {len(dropped)}:")
for row in dropped:
    print(f"  {row['sku']}  {row['name']}  — {row['why']}")

print("\nsave step reported:", run.output("save"))

sku     name                price  currency
TN-2P   Tent 2P            189.00  USD
SB-01   Sleeping bag        74.50  USD
HT-03   Head torch          21.00  USD

dropped 1:
  CS-77  Camp stove  — no price on the page

save step reported: {'rows': 3, 'path': '/home/username/code_projects/repos/browsergraph/notebooks/work/products.jsonl'}


In [8]:
print("files written:")
for art in run.artifacts:
    print(f"  {art.path:<34} {art.bytes:>8,} bytes  {art.digest[:18]}…")

files written:
  /home/username/code_projects/repos/browsergraph/notebooks/work/products.jsonl      218 bytes  sha256:15ae8e21fe5…


## Reading the saved file back

The file is the point. Here it is again, straight off disk.

In [9]:
saved = (WORK / "products.jsonl").read_text().splitlines()
print(f"{len(saved)} lines")
for line in saved:
    print(" ", line)

3 lines
  {"sku": "TN-2P", "name": "Tent 2P", "price": 189.0, "currency": "USD"}
  {"sku": "SB-01", "name": "Sleeping bag", "price": 74.5, "currency": "USD"}
  {"sku": "HT-03", "name": "Head torch", "price": 21.0, "currency": "USD"}


## Now do it against a live page

Same graph. Same checks. One different candidate for the `read` step.

This one really does go and fetch a page over the network. It is the honest
version of "browse and scrape", and it teaches something the fixture cannot.

In [10]:
LIVE_URL = "https://example.com"

live_route = dict(route, read="read.http")
plan_live = compile_route(bench, live_route)

print("same layers:", plan_live.layers == plan.layers)
print("different plan:", plan_live.digest != plan.digest)
print("effects now:", plan_live.effects)
print("deterministic:", plan_live.deterministic, "— a live page can change under you")

same layers: True
different plan: True
effects now: ('file.write', 'network.read')
deterministic: False — a live page can change under you


The plan says this run touches the network and is no longer deterministic. We
did not tell it that. It read it off the node we picked.

In [11]:
try:
    live = execute.run(plan_live, runtime, workspace=WORK)
    print(live.text())
    fetched = live.output("read")
    print(f"\nfetched {len(fetched):,} bytes from {LIVE_URL}")
except Exception as problem:
    live = None
    print("no network here:", problem)

plan plan:a1e978892e2fc01673b3a…
6 steps in 0.196s — ok
  ok   read             0.195s  read.http  [network.read]
  ok   parse            0.000s  parse.html
  ok   locate           0.000s  locate.fields
  ok   clean            0.000s  clean.values
  ok   check            0.000s  check.rows
  ok   save             0.000s  save.jsonl  [file.write]
  file /home/username/code_projects/repos/browsergraph/notebooks/work/products.jsonl  0 bytes  sha256:e3b0c4429…

fetched 559 bytes from https://example.com


## The result worth stopping on

The run finished. Look at what it actually produced.

In [12]:
if live is not None:
    kept_live = live.values.get(("check", "kept"), [])
    dropped_live = live.values.get(("check", "dropped"), [])
    print(f"steps ok: {live.ok}")
    print(f"products found: {len(kept_live)}")
    print(f"rows dropped:   {len(dropped_live)}")

steps ok: True
products found: 0
rows dropped:   0


Every step passed. Zero products came out.

That is the single most common way a scraper is broken, and it is why this
library exists. `example.com` has no `li.product` elements, so the locator found
nothing, and *finding nothing is not an error* — it is an empty list, which is a
perfectly good list.

Nothing crashed. A cron job running this would report success forever.

**A run that completed is not a run that worked.** The check has to be on the
result, not on whether the code threw.

In [13]:
def check_rows_strict(**kw):
    """Same check, plus one line: an empty result is a failure."""
    rows = kw["in"]
    if not rows:
        raise ValueError("the locator matched nothing — the selector is probably "
                         "wrong for this page")
    kept = [r for r in rows if r["price"] is not None]
    dropped = [dict(r, why="no price on the page") for r in rows if r["price"] is None]
    return {"kept": kept, "dropped": dropped}

strict = execute.Runtime(dict(runtime._functions))
strict.register("check.rows", check_rows_strict)

guarded = execute.run(plan_live, strict, workspace=WORK)
print("ok:", guarded.ok, "| stopped at:", guarded.stopped_at or "nowhere")
print(guarded.steps[-1].error)

ok: False | stopped at: check
check.rows: ValueError: the locator matched nothing — the selector is probably wrong for this page


Now it fails, at the step that noticed, with a sentence that says what to fix.

The fixture still works exactly as before — same graph, same runtime, one
different candidate for one step.

In [14]:
again = execute.run(plan, strict, workspace=WORK)
print("fixture run ok:", again.ok, "|", len(again.values[("check", "kept")]), "products")

fixture run ok: True | 3 products
